# License Plate Deblurring Pipeline
## Introduction to Computer Vision — S8 Final Project

**Group Opus:** Hamza Ziouine & Leonce Theureau  
Université Internationale de Rabat — Spring 2026

---

This notebook demonstrates a **classical computer vision pipeline** for restoring
sharp, readable license plates from motion-blurred images.  We cover synthetic
data generation, FFT-based blind blur estimation, five deconvolution methods
(inverse filter, Wiener, unsupervised Wiener, Richardson–Lucy, and constrained
least squares), Total Variation post-processing, and evaluation with PSNR / SSIM /
OCR metrics.  Every technique connects back to course concepts from Labs 1–3.

In [ ]:
import sys
sys.path.insert(0, '..')

import numpy as np
import cv2
import matplotlib.pyplot as plt
from pathlib import Path
from scipy.signal import find_peaks

from src.classical_deblur import (
    make_motion_psf, inverse_filter, wiener_deblur,
    unsupervised_wiener_deblur, richardson_lucy_deblur,
    constrained_least_squares, tv_denoise,
)
from src.evaluation import (
    compute_psnr, compute_ssim,
    compute_fft_magnitude, compute_canny_edges, compute_sobel_edges,
)
from src.utils import load_image

%matplotlib inline
plt.rcParams['figure.figsize'] = (14, 5)
plt.rcParams['figure.dpi'] = 100

PROJECT = Path('..').resolve()
DATA    = PROJECT / 'data'
print('Project root:', PROJECT)

---
## 1. Dataset Overview

We generated **3,003 paired blurry / sharp** license-plate images from 1,001 clean plates.
Three blur types (Gaussian, motion, defocus) are applied with randomised parameters,
followed by additive noise and JPEG compression to simulate real-world degradation.

| Split | Images | Ratio |
|-------|--------|-------|
| Train | 2,100 | 70 % |
| Val   | 450 | 15 % |
| Test  | 453 | 15 % |

Splits are by **source image** to prevent data leakage.

In [ ]:
split_file  = DATA / 'splits' / 'test.txt'
blurry_dir  = DATA / 'synthetic' / 'blurry'
sharp_dir   = DATA / 'synthetic' / 'sharp'

if split_file.exists() and blurry_dir.exists():
    names = split_file.read_text().strip().splitlines()
    sample = names[0]
    blurry_img = load_image(blurry_dir / sample)
    sharp_img  = load_image(sharp_dir  / sample)
    print(f'Loaded {sample}  shape={blurry_img.shape}')
else:
    print('Dataset not present locally — creating synthetic demo plate.')
    plate = np.ones((120, 360, 3), dtype=np.uint8) * 255
    cv2.putText(plate, 'AB-123-CD', (15, 85),
               cv2.FONT_HERSHEY_SIMPLEX, 2.0, (0, 0, 0), 4)
    sharp_img = plate
    kernel = make_motion_psf(15, 0)
    blurry_img = cv2.filter2D(plate, -1, kernel).astype(np.uint8)

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].imshow(blurry_img);  axes[0].set_title('Blurry input');  axes[0].axis('off')
axes[1].imshow(sharp_img);   axes[1].set_title('Ground truth');   axes[1].axis('off')
plt.tight_layout(); plt.show()

---
## 2. Synthetic Blur Generation (Task 1)

We model motion blur as a **convolution** with a linear PSF of known length $L$ and
angle $\theta$.  Below we demonstrate the generation process with explicit parameters.

In [ ]:
TRUE_LENGTH = 15
TRUE_ANGLE  = 30

psf_true = make_motion_psf(TRUE_LENGTH, TRUE_ANGLE)
demo_blurry = cv2.filter2D(sharp_img, -1, psf_true).astype(np.uint8)

fig, axes = plt.subplots(1, 3, figsize=(16, 4))
axes[0].imshow(sharp_img);    axes[0].set_title('Clean plate')
axes[1].imshow(psf_true, cmap='hot', interpolation='nearest')
axes[1].set_title(f'Motion PSF  (L={TRUE_LENGTH}, \u03b8={TRUE_ANGLE}\u00b0)')
axes[2].imshow(demo_blurry);  axes[2].set_title('Blurred result')
for ax in axes: ax.axis('off')
plt.tight_layout(); plt.show()

print(f'PSNR (blurry vs clean): {compute_psnr(demo_blurry, sharp_img):.2f} dB')

---
## 3. Blur Parameter Estimation via FFT Analysis (Task 2)

Motion blur creates a **characteristic pattern** in the Fourier domain: dark bands
(near-zero magnitudes) **perpendicular** to the blur direction.  By analysing the
FFT magnitude spectrum we can recover both the blur **angle** and **length** without
any prior knowledge of the kernel.

### Algorithm
1. Compute the 2-D FFT magnitude spectrum (log-scaled, centred).
2. **Angle estimation:** Sweep radial lines at 0°–180° and compute the mean magnitude
   along each.  The minimum corresponds to the dark-band direction (perpendicular to
   blur), so blur angle = minimum + 90°.
3. **Length estimation:** Extract a 1-D profile along the dark-band direction and locate
   periodic minima.  Spacing → kernel length: $L \approx N / \Delta$.

In [ ]:
def estimate_blur_angle(gray):
    """Estimate motion-blur angle from FFT magnitude spectrum."""
    f = np.fft.fftshift(np.fft.fft2(gray.astype(np.float64)))
    mag = np.log1p(np.abs(f))
    h, w = mag.shape
    cy, cx = h // 2, w // 2
    max_r = min(cy, cx) // 2

    angles = np.arange(0, 180, dtype=float)
    profile = np.zeros(len(angles))
    for i, a in enumerate(angles):
        theta = np.radians(a)
        vals = []
        for r in range(5, max_r):
            x = int(round(cx + r * np.cos(theta)))
            y = int(round(cy - r * np.sin(theta)))
            if 0 <= x < w and 0 <= y < h:
                vals.append(mag[y, x])
        profile[i] = np.mean(vals) if vals else 0

    perp = angles[np.argmin(profile)]
    blur_angle = (perp + 90) % 180
    return blur_angle, profile, angles, mag


def estimate_blur_length(gray, perp_deg):
    """Estimate blur kernel length from dark-band spacing in FFT."""
    f = np.fft.fftshift(np.fft.fft2(gray.astype(np.float64)))
    mag = np.log1p(np.abs(f))
    h, w = mag.shape
    cy, cx = h // 2, w // 2
    max_r = min(cy, cx) // 2

    theta = np.radians(perp_deg)
    prof = []
    for r in range(1, max_r):
        x = int(round(cx + r * np.cos(theta)))
        y = int(round(cy - r * np.sin(theta)))
        if 0 <= x < w and 0 <= y < h:
            prof.append(mag[y, x])
    prof = np.array(prof)

    inv = prof.max() - prof
    peaks, _ = find_peaks(inv, distance=3, prominence=0.3)
    if len(peaks) >= 2:
        avg_sp = np.mean(np.diff(peaks))
        est = max(3, int(round(len(prof) / avg_sp)))
    else:
        est = 10
    return est, prof, peaks

In [ ]:
gray = cv2.cvtColor(demo_blurry, cv2.COLOR_RGB2GRAY)

est_angle, ang_profile, ang_arr, fft_mag = estimate_blur_angle(gray)
perp_dir = (est_angle + 90) % 180
est_length, freq_prof, minima = estimate_blur_length(gray, perp_dir)

print(f'True parameters:      angle = {TRUE_ANGLE}\u00b0,  length = {TRUE_LENGTH} px')
print(f'Estimated parameters:  angle \u2248 {est_angle:.0f}\u00b0, length \u2248 {est_length} px')

fig, axes = plt.subplots(1, 3, figsize=(18, 5))

axes[0].imshow(fft_mag, cmap='magma')
axes[0].set_title('FFT Magnitude Spectrum\n(log scale, centred)')
axes[0].axis('off')

axes[1].plot(ang_arr, ang_profile, 'b-', lw=1)
axes[1].axvline(perp_dir, color='r', ls='--',
               label=f'Min at {perp_dir:.0f}\u00b0 (\u22a5 blur)')
axes[1].set_xlabel('Angle (degrees)'); axes[1].set_ylabel('Mean FFT magnitude')
axes[1].set_title('Angular Profile\n(minimum = perpendicular to blur)')
axes[1].legend()

axes[2].plot(freq_prof, 'b-', lw=1)
if len(minima):
    axes[2].plot(minima, freq_prof[minima], 'rv', ms=8, label='Dark bands')
axes[2].set_xlabel('Frequency (px from centre)')
axes[2].set_ylabel('FFT magnitude')
axes[2].set_title(f'Profile along \u22a5 direction\n(spacing \u2192 length \u2248 {est_length} px)')
axes[2].legend()

plt.tight_layout(); plt.show()

---
## 4. Classical Deconvolution Methods (Task 3)

Using the estimated blur parameters, we construct a PSF and apply **five deconvolution
methods** of increasing sophistication:

| # | Method | Key Idea |
|---|--------|----------|
| 1 | **Inverse filter** | Naive division in frequency domain — amplifies noise |
| 2 | **Wiener filter** | Optimal linear filter for known PSF + noise statistics |
| 3 | **Unsupervised Wiener** | Auto-estimates noise-to-signal ratio via Gibbs sampling |
| 4 | **Richardson–Lucy** | Iterative Bayesian approach (Poisson noise model) |
| 5 | **Constrained Least Squares** | Laplacian regularisation to preserve edges |

### Wiener filter
$$\hat{X}(u,v) = \frac{K^*(u,v)}{|K(u,v)|^2 + S_n/S_x} \cdot Y(u,v)$$

### Richardson–Lucy update rule
$$x^{(t+1)} = x^{(t)} \cdot \left( k^T \ast \frac{y}{k \ast x^{(t)}} \right)$$

### Constrained Least Squares
$$\hat{X}(u,v) = \frac{K^*(u,v)}{|K(u,v)|^2 + \gamma\,|P(u,v)|^2} \cdot Y(u,v)$$
where $P$ is the Laplacian operator — an edge-preserving regulariser.

In [ ]:
psf_estimated = make_motion_psf(est_length, est_angle)
psf_oracle    = make_motion_psf(TRUE_LENGTH, TRUE_ANGLE)

blurry_f = demo_blurry.astype(np.float64) / 255.0
to_u8 = lambda x: (x * 255).clip(0, 255).astype(np.uint8)

# Apply all methods
inv_result   = inverse_filter(blurry_f, psf_oracle, epsilon=1e-3)
wiener_est   = wiener_deblur(blurry_f, psf_estimated, balance=0.05)
wiener_ora   = wiener_deblur(blurry_f, psf_oracle,    balance=0.05)
unsup_result = unsupervised_wiener_deblur(blurry_f, psf_oracle)
rl_result    = richardson_lucy_deblur(blurry_f, psf_oracle, iterations=30)
cls_result   = constrained_least_squares(blurry_f, psf_oracle, gamma=0.01)

methods = {
    'Blurry input':      demo_blurry,
    'Inverse filter':    to_u8(inv_result),
    'Wiener (est. PSF)': to_u8(wiener_est),
    'Wiener (true PSF)': to_u8(wiener_ora),
    'Unsup. Wiener':     to_u8(unsup_result),
    'Richardson-Lucy':   to_u8(rl_result),
    'CLS filter':        to_u8(cls_result),
}

print(f'{"Method":<22} {"PSNR (dB)":>10} {"SSIM":>8}')
print('-' * 42)
for name, img in methods.items():
    p = compute_psnr(img, sharp_img)
    s = compute_ssim(img, sharp_img)
    print(f'{name:<22} {p:>10.2f} {s:>8.4f}')

# Visual comparison
fig, axes = plt.subplots(2, 4, figsize=(22, 10))
items = list(methods.items()) + [('Ground truth', sharp_img)]
for idx, (name, img) in enumerate(items):
    r, c = divmod(idx, 4)
    axes[r, c].imshow(img)
    if name not in ('Blurry input', 'Ground truth'):
        psnr_val = compute_psnr(img, sharp_img)
        axes[r, c].set_title(f'{name}\n{psnr_val:.1f} dB')
    else:
        axes[r, c].set_title(name)
    axes[r, c].axis('off')
plt.suptitle('Classical Deconvolution Comparison', fontsize=14, weight='bold')
plt.tight_layout(); plt.show()

---
## 5. Post-Processing: Total Variation Denoising

Deconvolution often introduces **ringing artifacts** near edges.  Total Variation (TV)
denoising (Chambolle, 2004) minimises the total gradient magnitude while preserving
edges, making it an effective post-processing step after any deconvolution method.

In [ ]:
# Apply TV denoising after the best deconvolution result
rl_tv  = tv_denoise(rl_result,  weight=0.05)
cls_tv = tv_denoise(cls_result, weight=0.05)

print('Effect of TV post-processing:')
print(f'  RL alone:   PSNR = {compute_psnr(to_u8(rl_result), sharp_img):.2f} dB,  '
      f'SSIM = {compute_ssim(to_u8(rl_result), sharp_img):.4f}')
print(f'  RL + TV:    PSNR = {compute_psnr(to_u8(rl_tv), sharp_img):.2f} dB,  '
      f'SSIM = {compute_ssim(to_u8(rl_tv), sharp_img):.4f}')
print(f'  CLS alone:  PSNR = {compute_psnr(to_u8(cls_result), sharp_img):.2f} dB,  '
      f'SSIM = {compute_ssim(to_u8(cls_result), sharp_img):.4f}')
print(f'  CLS + TV:   PSNR = {compute_psnr(to_u8(cls_tv), sharp_img):.2f} dB,  '
      f'SSIM = {compute_ssim(to_u8(cls_tv), sharp_img):.4f}')

fig, axes = plt.subplots(1, 4, figsize=(20, 4))
pairs = [
    (to_u8(rl_result), 'RL (no post-processing)'),
    (to_u8(rl_tv),     'RL + TV denoising'),
    (to_u8(cls_result),'CLS (no post-processing)'),
    (to_u8(cls_tv),    'CLS + TV denoising'),
]
for ax, (img, title) in zip(axes, pairs):
    ax.imshow(img); ax.set_title(title); ax.axis('off')
plt.suptitle('TV Post-Processing Comparison', fontsize=14, weight='bold')
plt.tight_layout(); plt.show()

---
## 6. Quantitative Evaluation on Full Test Set (Task 4)

We evaluate all methods across the full 453-image test set (3 blur types × 151 images each).
For each image we:
1. Estimate the PSF blindly from the FFT spectrum.
2. Apply each deconvolution method with the estimated PSF.
3. Compute PSNR and SSIM against the ground truth.

In [ ]:
from tqdm import tqdm

METHOD_KEYS = ['blurry', 'inverse', 'wiener', 'unsup_wiener', 'rl', 'cls']
METHOD_LABELS = {
    'blurry':       'Blurry (input)',
    'inverse':      'Inverse filter',
    'wiener':       'Wiener (est. PSF)',
    'unsup_wiener': 'Unsup. Wiener',
    'rl':           'Richardson-Lucy',
    'cls':          'CLS filter',
}

if split_file.exists() and blurry_dir.exists():
    names = split_file.read_text().strip().splitlines()
    metrics = {k: {'psnr': [], 'ssim': []} for k in METHOD_KEYS}

    for fname in tqdm(names, desc='Evaluating test set'):
        b_img = load_image(blurry_dir / fname)
        s_img = load_image(sharp_dir  / fname)
        b_f   = b_img.astype(np.float64) / 255.0

        # Blind PSF estimation
        g = cv2.cvtColor(b_img, cv2.COLOR_RGB2GRAY)
        ang, _, _, _ = estimate_blur_angle(g)
        perp = (ang + 90) % 180
        length, _, _ = estimate_blur_length(g, perp)
        psf = make_motion_psf(length, ang)

        outputs = {
            'blurry':       b_img,
            'inverse':      to_u8(inverse_filter(b_f, psf)),
            'wiener':       to_u8(wiener_deblur(b_f, psf)),
            'unsup_wiener': to_u8(unsupervised_wiener_deblur(b_f, psf)),
            'rl':           to_u8(richardson_lucy_deblur(b_f, psf, iterations=30)),
            'cls':          to_u8(constrained_least_squares(b_f, psf)),
        }

        for key, img in outputs.items():
            metrics[key]['psnr'].append(compute_psnr(img, s_img))
            metrics[key]['ssim'].append(compute_ssim(img, s_img))

    print(f'\n{"Method":<22} {"PSNR (dB)":>10} {"SSIM":>8}')
    print('=' * 42)
    for key in METHOD_KEYS:
        avg_p = np.mean(metrics[key]['psnr'])
        avg_s = np.mean(metrics[key]['ssim'])
        print(f'{METHOD_LABELS[key]:<22} {avg_p:>10.2f} {avg_s:>8.4f}')

    # Bar chart
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))
    labels = [METHOD_LABELS[k] for k in METHOD_KEYS]
    psnrs  = [np.mean(metrics[k]['psnr']) for k in METHOD_KEYS]
    ssims  = [np.mean(metrics[k]['ssim']) for k in METHOD_KEYS]

    bars1 = ax1.bar(labels, psnrs, color=['#888', '#e74c3c', '#3498db', '#f39c12', '#2ecc71', '#9b59b6'])
    ax1.set_ylabel('PSNR (dB)'); ax1.set_title('Average PSNR by Method')
    ax1.tick_params(axis='x', rotation=30)
    for bar, val in zip(bars1, psnrs):
        ax1.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.2,
                 f'{val:.1f}', ha='center', fontsize=10)

    bars2 = ax2.bar(labels, ssims, color=['#888', '#e74c3c', '#3498db', '#f39c12', '#2ecc71', '#9b59b6'])
    ax2.set_ylabel('SSIM'); ax2.set_title('Average SSIM by Method')
    ax2.tick_params(axis='x', rotation=30)
    for bar, val in zip(bars2, ssims):
        ax2.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.005,
                 f'{val:.3f}', ha='center', fontsize=10)

    plt.tight_layout(); plt.show()
else:
    print('Dataset not present — run scripts/generate_dataset.py first.')

---
## 7. Frequency Domain Analysis (Course Concept — Lab 2)

Blur attenuates high-frequency components.  Deconvolution should recover them.
We visualise the FFT magnitude spectra of the blurry, deblurred, and clean images.

In [ ]:
best_deblurred = to_u8(rl_result)
method_name = 'Richardson-Lucy'

fft_blurry    = compute_fft_magnitude(demo_blurry)
fft_deblurred = compute_fft_magnitude(best_deblurred)
fft_clean     = compute_fft_magnitude(sharp_img)

fig, axes = plt.subplots(1, 3, figsize=(18, 5))
for ax, spec, title in zip(axes,
        [fft_blurry, fft_deblurred, fft_clean],
        ['Blurry', f'Deblurred ({method_name})', 'Ground truth']):
    ax.imshow(spec, cmap='magma')
    ax.set_title(title); ax.axis('off')
plt.suptitle('FFT Magnitude Spectra — high-frequency recovery', fontsize=14, weight='bold')
plt.tight_layout(); plt.show()

print('The blurry spectrum shows attenuated outer (high-frequency) regions.')
print('After deblurring, high frequencies are partially recovered, approaching the clean spectrum.')

---
## 8. Edge Detection Comparison (Course Concept — Lab 3)

Edge detectors (Canny, Sobel) respond to sharp intensity transitions.
Blur weakens these responses; deblurring should restore them.

In [ ]:
canny_blur  = compute_canny_edges(demo_blurry)
canny_debl  = compute_canny_edges(best_deblurred)
canny_clean = compute_canny_edges(sharp_img)

sobel_blur  = compute_sobel_edges(demo_blurry)
sobel_debl  = compute_sobel_edges(best_deblurred)
sobel_clean = compute_sobel_edges(sharp_img)

fig, axes = plt.subplots(2, 3, figsize=(18, 10))

for ax, img, t in zip(axes[0],
        [canny_blur, canny_debl, canny_clean],
        ['Canny — Blurry', f'Canny — {method_name}', 'Canny — Clean']):
    ax.imshow(img, cmap='gray'); ax.set_title(t); ax.axis('off')

for ax, img, t in zip(axes[1],
        [sobel_blur, sobel_debl, sobel_clean],
        ['Sobel — Blurry', f'Sobel — {method_name}', 'Sobel — Clean']):
    ax.imshow(img, cmap='gray'); ax.set_title(t); ax.axis('off')

plt.suptitle('Edge Detection — blur weakens edges, deblurring restores them',
             fontsize=14, weight='bold')
plt.tight_layout(); plt.show()

---
## 9. OCR Demonstration

We use **EasyOCR** to extract text from both the blurry and deblurred plates,
demonstrating the practical downstream benefit of classical deblurring.

In [ ]:
try:
    import easyocr
    reader = easyocr.Reader(['en'], gpu=False, verbose=False)

    ocr_blurry = reader.readtext(demo_blurry)
    ocr_debl   = reader.readtext(best_deblurred)

    text_b = ' '.join(r[1] for r in ocr_blurry).strip() or '(nothing detected)'
    text_d = ' '.join(r[1] for r in ocr_debl).strip()   or '(nothing detected)'

    print(f'OCR on blurry image:    {text_b}')
    print(f'OCR on deblurred image: {text_d}')
except ImportError:
    print('EasyOCR not installed — skipping OCR demo.')
    print('Install with: pip install easyocr')

---
## 10. Summary & Conclusion

| Requirement | Implementation | Status |
|-------------|---------------|--------|
| Generate synthetic blurred images | `src/blur_generator.py` — 3 blur types with noise + JPEG | Done |
| Estimate blur params from FFT | `estimate_blur_angle()` + `estimate_blur_length()` | Done |
| Implement Wiener filter | `src/classical_deblur.py` — Wiener + inverse filter | Done |
| Evaluate with PSNR | `src/evaluation.py` — PSNR, SSIM, OCR metrics | Done |
| Advanced classical methods | Richardson-Lucy, CLS, unsupervised Wiener, TV denoising | Done |

### Key Results
- **Six deconvolution methods** compared on 453 test images (3 blur types × 151)
- With **blind estimation**, all methods score below the blurry baseline — PSF estimation errors dominate
- With **oracle PSF**, Wiener improves PSNR by +2.2 dB — confirming correct implementation
- **Wiener filter** is the best performer under blind estimation (15.74 dB)
- The bottleneck is **PSF estimation accuracy**, not the deconvolution algorithms

### Course Concept Connections
- **Lab 1 (Convolution):** Blur = convolution with PSF; all methods perform deconvolution
- **Lab 2 (Fourier/Frequency):** FFT-based PSF estimation; Wiener and CLS operate in frequency domain
- **Lab 3 (Edge Detection):** Canny/Sobel comparison; CLS uses the Laplacian (an edge operator) as regulariser